# Reproducible test evaluation in Colab

Run this notebook in order: install one identified commit, connect Drive and Secrets, check all four models, run ten fixed development items, inspect outputs and measured usage, then run all 395 test items. Both generation and selection are retained for each LLM. Test includes natural and AI-authored text; inspect source groups separately.

Use a **free GPU** for Qwen if available; no paid runtime is required or purchased here. Account access and live Colab execution must be verified. See [the operating guide](../docs/reproducible_evaluation.md) for the upload route, keys and recovery. Historical outputs are never overwritten. Retraining is not part of this notebook.

## 1. Install the identified source

Fill `CODE_REVISION` with the reviewed 40-character commit. For the default archive route, upload its `git archive --format=tar` file with Colab's Files panel, set `ARCHIVE_PATH` and copy the independently supplied SHA-256. This works without publishing the branch. The optional `git` route fetches only the exact commit; it requires that commit to be accessible on the remote. The package is installed normally so the running kernel can import it immediately. After changing commits or if Colab reports already-loaded dependency changes, restart the runtime and rerun setup before continuing.

In [ ]:
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import tarfile
import tempfile
import time
import urllib.request

CODE_REVISION = ""  # Reviewed 40-character commit; never main/HEAD/a branch.
SOURCE_MODE = "archive"  # "archive" or "git"
ARCHIVE_PATH = Path("/content/evaluation-release.tar")
ARCHIVE_SHA256 = ""  # Supplied with the release, not inferred from the upload.
REPOSITORY_URL = "https://github.com/BenCarmel123/hebrew-acronym-disambiguation.git"
if not re.fullmatch(r"[0-9a-f]{40}", CODE_REVISION):
    raise ValueError("Set the reviewed full commit SHA before installation")
REPO_DIR = Path("/content/hebrew-eval-" + CODE_REVISION)
REPO_DIR.mkdir(parents=True, exist_ok=True)
if SOURCE_MODE == "archive":
    if not re.fullmatch(r"[0-9a-f]{64}", ARCHIVE_SHA256):
        raise ValueError("Set the release archive SHA-256")
    archive_bytes = ARCHIVE_PATH.read_bytes()
    if hashlib.sha256(archive_bytes).hexdigest() != ARCHIVE_SHA256:
        raise ValueError("Uploaded archive differs from the reviewed release")
    archive_revision = subprocess.check_output(
        ["git", "get-tar-commit-id"], input=archive_bytes).decode().strip()
    if archive_revision != CODE_REVISION:
        raise ValueError("Archive commit differs from CODE_REVISION")
    with tarfile.open(ARCHIVE_PATH, "r:") as archive:
        archive.extractall(REPO_DIR, filter="data")
elif SOURCE_MODE == "git":
    subprocess.run(["git", "init", str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1",
                    REPOSITORY_URL, CODE_REVISION], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
    actual = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
    if actual != CODE_REVISION:
        raise ValueError("Fetched source differs from CODE_REVISION")
else:
    raise ValueError("Choose archive or git")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_DIR)], check=True)
# Regular installation is immediately importable in this running notebook kernel.
# Check this project's pins; Colab can contain unrelated preinstalled conflicts.
for line in (REPO_DIR / "requirements.txt").read_text().splitlines():
    requirement = line.strip()
    if requirement and not requirement.startswith("#"):
        package_name, required_version = requirement.split("==", 1)
        if importlib.metadata.version(package_name) != required_version:
            raise RuntimeError(f"Required package version mismatch: {package_name}")
pip_report = subprocess.run([sys.executable, "-m", "pip", "check"],
                            capture_output=True, text=True)
if pip_report.returncode:
    print("Colab environment dependency report (inspect separately from project pins):")
    print(pip_report.stdout or pip_report.stderr)
print("Installed source commit:", CODE_REVISION)

## 2. Connect persistent storage and credentials

Add `GEMINI_API_KEY`, `OPENAI_API_KEY` and `ANTHROPIC_API_KEY` in Colab **Secrets** (key icon), and enable notebook access. Never put key values in cells, output or chat. Keep the same `RUN_NAME` after a disconnect. Use a new name only for an intentionally new run and include previous expenditure in `PRIOR_SPEND_ILS`.

In [ ]:
from google.colab import drive, userdata
import importlib.metadata
import platform
import torch
from hebrew_acronyms.test_evaluation import (
    prepare_evaluation, run_evaluation, summarize_evaluation, estimate_cost,
)
from hebrew_acronyms.models.qwen.eval import inspect_ollama
from hebrew_acronyms.models.openai.eval import inspect_openai_model
from hebrew_acronyms.models.anthropic.eval import inspect_anthropic_model

RUN_NAME = "evaluation-20261009-01"
PRIOR_SPEND_ILS = 0.0  # All earlier API attempts outside this pilot/full pair.
BUDGET_ILS = 100.0
ILS_PER_USD_ALLOWANCE = 4.0  # Conservative conversion incl. fee/tax margin, not quoted FX.
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME):
    raise ValueError("Use a simple stable run name")
if not 0 <= PRIOR_SPEND_ILS < BUDGET_ILS:
    raise ValueError("Prior expenditure must leave a positive budget")
drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/NLP/evaluation-runs") / RUN_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PILOT_DIR = OUTPUT_ROOT / "dev-pilot"
TEST_DIR = OUTPUT_ROOT / "full-test"
for secret_name in ("GEMINI_API_KEY", "OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
    try:
        secret_value = userdata.get(secret_name)
        if not secret_value:
            raise ValueError("empty")
        os.environ[secret_name] = secret_value
    except Exception:
        raise RuntimeError(f"Add {secret_name} in Colab Secrets and enable notebook access") from None
del secret_value
ENVIRONMENT = {
    "code_revision": CODE_REVISION, "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name) for name in
                 ("torch", "transformers", "requests", "numpy")},
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "archive_sha256": ARCHIVE_SHA256 if SOURCE_MODE == "archive" else None,
}
with (OUTPUT_ROOT / f"environment-{time.time_ns()}.json").open("x") as stream:
    json.dump(ENVIRONMENT, stream, indent=2)
print(ENVIRONMENT)
print("Persistent outputs:", OUTPUT_ROOT)

## 3. Start and inspect the exact Qwen model

The explicit installation downloads Ollama and `qwen2.5:7b` into the temporary Colab VM. The run records its actual version, model digest, quantization, template and stop parameters. Review these before the pilot. Do not substitute a different model when access, resources or output quality are poor.

In [ ]:
QWEN_MODEL = "qwen2.5:7b"
QWEN_OPTIONS = {"temperature": 0, "seed": 42, "num_predict": 512}
OLLAMA_URL = "http://localhost:11434"
if shutil.which("ollama") is None:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"], check=True)
    installer = Path("/content/ollama-install.sh")
    urllib.request.urlretrieve("https://ollama.com/install.sh", installer)
    subprocess.run(["sh", str(installer)], check=True)
import requests
try:
    requests.get(OLLAMA_URL + "/api/version", timeout=5).raise_for_status()
except requests.RequestException:
    ollama_log = (OUTPUT_ROOT / f"ollama-{time.time_ns()}.log").open("w")
    ollama_process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)
    for _ in range(30):
        time.sleep(1)
        try:
            requests.get(OLLAMA_URL + "/api/version", timeout=2).raise_for_status()
            break
        except requests.RequestException:
            pass
    else:
        raise RuntimeError("Ollama did not start; inspect its saved log")
subprocess.run(["ollama", "pull", QWEN_MODEL], check=True)
QWEN_IDENTITY = inspect_ollama(QWEN_MODEL, options=QWEN_OPTIONS)
print(json.dumps(QWEN_IDENTITY, ensure_ascii=False, indent=2))

## 4. Check accounts and freeze the model settings

Availability checks below do not generate text. They must succeed for the exact approved IDs. The pilot then checks actual generation access, completion reasons and model versions. Gemini uses low thinking; Claude uses adaptive thinking with low effort. Output caps include provider-billed thinking where applicable. No automatic fallback is configured.

Prices are USD per million tokens, checked 9 October 2026: [Gemini](https://ai.google.dev/gemini-api/docs/pricing), [OpenAI](https://developers.openai.com/api/docs/models/gpt-4.1-mini), [Claude](https://platform.claude.com/docs/en/models/haiku-5-5/overview). The cost review uses provider-reported usage; it is a tariff estimate until reconciled with billing. Cached input is conservatively charged at the full rate.

In [ ]:
SYSTEMS = [
    {"name": "qwen", "provider": "qwen", "model": QWEN_MODEL,
     "settings": {"expected_digest": QWEN_IDENTITY["digest"], "base_url": OLLAMA_URL,
                  "timeout": 120, "options": QWEN_OPTIONS}},
    {"name": "gemini", "provider": "gemini", "model": "gemini-3.8-flash",
     "settings": {"timeout": 120, "generation_config": {
         "maxOutputTokens": 1024,
         "thinkingConfig": {"thinkingLevel": "low"}}}},
    {"name": "openai", "provider": "openai", "model": "gpt-4.1-mini-2025-04-14",
     "settings": {"timeout": 120, "max_output_tokens": 512, "temperature": 0}},
    {"name": "anthropic", "provider": "anthropic", "model": "claude-haiku-5-5",
     "settings": {"timeout": 120, "max_output_tokens": 1024, "effort": "low"}},
]
RATES = {"qwen": {"input": 0.0, "output": 0.0},
         "gemini": {"input": 0.75, "output": 3.75},
         "openai": {"input": 0.40, "output": 1.60},
         "anthropic": {"input": 0.10, "output": 0.50}}
AVAILABILITY = {
    "openai": inspect_openai_model(model=SYSTEMS[2]["model"]),
    "anthropic": inspect_anthropic_model(model=SYSTEMS[3]["model"]),
}
gemini_info = requests.get(
    "https://generativelanguage.googleapis.com/v1beta/models/" + SYSTEMS[1]["model"],
    headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]}, timeout=30)
if gemini_info.status_code != 200:
    raise RuntimeError(f"Exact Gemini model unavailable: HTTP {gemini_info.status_code}")
AVAILABILITY["gemini"] = {key: gemini_info.json().get(key) for key in
                          ("name", "version", "supportedGenerationMethods")}
if "generateContent" not in (AVAILABILITY["gemini"]["supportedGenerationMethods"] or []):
    raise RuntimeError("Gemini model does not advertise generateContent")
if any(info.get("status") != "available" for name, info in AVAILABILITY.items() if name != "gemini"):
    raise RuntimeError("Exact model lookup failed; inspect account/model access without substitution")
print(json.dumps(AVAILABILITY, indent=2))
DEV_PATH = REPO_DIR / "data/splits/dev_items.csv"
TEST_PATH = REPO_DIR / "data/splits/test_items.csv"
RUN_METADATA = {"qwen_inspection": QWEN_IDENTITY, "rates_usd_per_million": RATES,
                "conversion_allowance_ils_per_usd": ILS_PER_USD_ALLOWANCE,
                "prior_spend_ils": PRIOR_SPEND_ILS}

## 5. Run the fixed development pilot

The pilot is the first ten stored development rows, in file order: 80 logical responses across four models and two tasks. The runner persists the identified request before calling a model and each outcome immediately to Drive. At most two attempts per request are permitted; transient failures may retry, incorrect answers do not. A crashed request remains ambiguous and is not silently resent.

The budget reserve below is an intentionally conservative allowance for each potentially billed request. Completed usage drives the simple full-run estimate in the next cell. Resume the same directory after a restart; a data or configuration mismatch must fail.

In [ ]:
# Upper input allowance: prompt UTF-8 bytes plus provider-envelope margin.
from hebrew_acronyms.models.common.pairs import load_rows
from hebrew_acronyms.models.common.eval import build_generate_prompt, build_select_prompt
all_rows = load_rows(DEV_PATH) + load_rows(TEST_PATH)
max_prompt_bytes = max(len(prompt.encode("utf-8")) for row in all_rows for prompt in (
    build_generate_prompt(row["acronym"], row["sentence"]),
    build_select_prompt(row["acronym"], row["sentence"], row["candidates"].split("|")),
))
input_token_allowance = max_prompt_bytes + 1024
if input_token_allowance >= 100_000:
    raise ValueError("Recheck long-context price tier before any calls")
OUTPUT_CAPS = {"qwen": 512, "gemini": 1024, "openai": 512, "anthropic": 1024}
RESERVE_PER_CALL_USD = {
    name: (input_token_allowance * rate["input"] + OUTPUT_CAPS[name] * rate["output"]) / 1_000_000
    for name, rate in RATES.items()
}
PILOT_BUDGET_USD = (BUDGET_ILS - PRIOR_SPEND_ILS) / ILS_PER_USD_ALLOWANCE
pilot_manifest = prepare_evaluation(
    DEV_PATH, PILOT_DIR, cohort="dev_pilot", systems=SYSTEMS,
    code_revision=CODE_REVISION, seed=42, max_attempts=2, max_calls=160,
    metadata=RUN_METADATA, reserve_per_call_usd=RESERVE_PER_CALL_USD,
    budget_usd=PILOT_BUDGET_USD, rates_usd_per_million=RATES,
)
pilot_summary = run_evaluation(PILOT_DIR, code_revision=CODE_REVISION)
print({key: pilot_summary[key] for key in
       ("n_items", "n_records", "n_calls", "n_completed", "n_pending", "n_ambiguous")})

## 6. Inspect pilot responses, usage and projected cost

Inspect every pilot response, especially role/template leakage, truncation, finish reasons and returned identities. A semantically wrong response remains a valid observation; do not tune on test results or require high pilot accuracy. Confirm the pilot ran as intended by setting `PILOT_INSPECTED = True` only after reading this output. If a model is unavailable or consistently cut off, stop and resolve that issue with a new identified pilot.

The estimate scales measured calls/tokens/time from 10 to 395 items and adds 25% cost reserve. Total spending includes this pilot, earlier runs and retries. It must fit 100 ILS. No full-test API call precedes this gate.

In [ ]:
pilot_summary = summarize_evaluation(PILOT_DIR)
print(json.dumps(pilot_summary["by_system_task"], ensure_ascii=False, indent=2))
print(json.dumps(pilot_summary["usage"], ensure_ascii=False, indent=2))
for record in pilot_summary["records"]:
    print(json.dumps(record, ensure_ascii=False))
cost_review = estimate_cost(pilot_summary, rates_usd_per_million=RATES,
                            full_items=395, reserve_fraction=0.25)
projected_total_ils = PRIOR_SPEND_ILS + ILS_PER_USD_ALLOWANCE * (
    cost_review["pilot_usd"] + cost_review["projected_full_with_reserve_usd"])
print(json.dumps(cost_review, indent=2))
print("Model | calls | input | output | thinking | pilot ILS | full ILS + 25% | full minutes")
for name, usage in pilot_summary["usage"].items():
    costs = cost_review["by_system"][name]
    seconds = pilot_summary["timings_seconds"][name]
    print(name, usage["attempts_with_usage"] + usage["attempts_without_usage"],
          usage["input_tokens"], usage["output_tokens"], usage["thinking_tokens"],
          round(costs["pilot_usd"] * ILS_PER_USD_ALLOWANCE, 3),
          round(costs["projected_full_usd"] * 1.25 * ILS_PER_USD_ALLOWANCE, 2),
          round(seconds * 395 / 10 / 60, 1), sep=" | ")
print("Projected total including pilot and reserve (ILS):", round(projected_total_ils, 2))
PILOT_INSPECTED = False  # Set True after the output/identity/completion inspection above.
PILOT_PASSED = (pilot_summary["n_items"] == 10 and pilot_summary["n_records"] == 80
                and pilot_summary["n_completed"] == 80 and pilot_summary["n_pending"] == 0
                and pilot_summary["n_ambiguous"] == 0
                and all(r["response_metadata"].get("identity_status") != "unverified"
                        for r in pilot_summary["records"]) and PILOT_INSPECTED)
if not PILOT_PASSED or projected_total_ils > BUDGET_ILS:
    raise RuntimeError("Full test is gated: complete the pilot inspection and budget review first")

## 7. Run or resume the full test

Uses all 395 unchanged test items and the same settings: 3,160 logical responses. Re-run this cell to continue the same run. Only pending/retry-eligible work is submitted; completed, incomplete and ambiguous answers remain recorded. Never launch two sessions writing the same run folder. Stop if the remaining budget is insufficient; do not silently omit a model or change the benchmark.

In [ ]:
if not PILOT_PASSED or projected_total_ils > BUDGET_ILS:
    raise RuntimeError("A successful inspected pilot within budget is required")
# Charge known pilot usage and reserve unknown attempts; never recover budget by losing a log.
pilot_reserved_usd = pilot_summary["charged_or_reserved_usd"]
full_budget_usd = PILOT_BUDGET_USD - pilot_reserved_usd
if full_budget_usd <= 0:
    raise RuntimeError("No API budget remains")
test_manifest = prepare_evaluation(
    TEST_PATH, TEST_DIR, cohort="full_test", systems=SYSTEMS,
    code_revision=CODE_REVISION, seed=42, max_attempts=2, max_calls=6320,
    metadata={**RUN_METADATA, "pilot_run": str(PILOT_DIR), "pilot_cost_review": cost_review},
    reserve_per_call_usd=RESERVE_PER_CALL_USD, budget_usd=full_budget_usd,
    rates_usd_per_million=RATES,
)
test_summary = run_evaluation(TEST_DIR, code_revision=CODE_REVISION)
print({key: test_summary[key] for key in
       ("n_items", "n_records", "n_calls", "n_completed", "n_pending", "n_ambiguous")})

## 8. Preserve the baseline comparison and checkpoint route

The deterministic baselines are retained: uniform-random expectation, most-frequent, most-mined and oracle. The trained DictaBERT and untrained similarity baseline remain separate systems; no retraining occurs here. Checkpoint reuse must identify the full weights and tokenizer/config snapshot. Historical training seed, selected epoch and exact training data are not recovered from a state dictionary.

The existing package [Colab checkpoint adapter](../src/hebrew_acronyms/models/dictabert_cross_encoder/colab.py) accepts the authorized weights SHA-256 `23bbff0324a7ce4d9d4a24a9ebfb87a4f6ac296bf1f1ce1bd25126c09260267f`. Supply an existing local snapshot and checkpoint from Drive. Do not use the old notebook's inferred seed or mutable model download as training provenance. The [checkpoint guide](../docs/checkpoints.md) describes the preserved inference workflow.

**Encoder execution limitation:** the historical test CSV lacks explicit target spans required by the strict package encoder. Preserve these arms, but qualify the target mapping before fresh checkpoint evaluation; do not silently take the first matching occurrence. Historical encoder results remain separately identifiable. This notebook does not claim new encoder predictions.

In [ ]:
from hebrew_acronyms.test_baselines import run_test_baselines
baseline_summary = run_test_baselines(
    TEST_PATH, REPO_DIR / "data/mined/candidate_table.csv",
    OUTPUT_ROOT / "baselines", code_revision=CODE_REVISION,
)
print(baseline_summary["summary"])

## 9. Inspect and export saved evidence

`manifest.json` identifies code, inputs, complete prompts, candidate orders and settings. `attempts.jsonl` preserves attempts; `summary.json` includes coverage and raw records. Keep the complete Drive folder. Missing/incomplete responses stay in the denominator. Automatic letter parsing and historical string scoring are not human generation judgments.

The methods/analysis entry point remains [experimental_study.ipynb](experimental_study.ipynb). Do not copy a provisional aggregate into the manuscript: use an explicitly selected run and retain source groups, model versions and human-judgment provenance. The current dev paper exporter is not a full-test exporter. Downloading the saved archive below makes a local copy; it does not publish results or send them to another person.

In [ ]:
from google.colab import files
saved_summary = summarize_evaluation(TEST_DIR)
print(json.dumps(saved_summary["by_system_task"], ensure_ascii=False, indent=2))
print("Inspect saved records at", TEST_DIR / "summary.json")
archive_path = shutil.make_archive("/content/" + RUN_NAME, "zip", OUTPUT_ROOT)
files.download(archive_path)